# Evaluate the released model on unseen data

Loads `esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b` from the Hub (no
retraining) and scores it on two things:

1. the committed held-out **test set** (reproduces the paper number ~0.975);
2. a **freshly generated, never-seen** test set (new generator seed,
   deduplicated against the model's training data) — a real generalization
   check on new draws from the same distribution.

**Important:** the released model was trained on ALL 5 domains, so this is an
*in-distribution* generalization test (unseen examples, seen distribution).
A true out-of-distribution / domain-hold-out test needs the model trained
*without* the test domains — that is `colab_overfitting_checks.ipynb`, not
this notebook.

Runtime → T4/L4 GPU → Run all.

In [ ]:
# Idempotent clone + absolute cd (prevents nested-directory issues).
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
print('cwd:', os.getcwd())
import torch; assert torch.cuda.is_available()
MODEL = "esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b"


## 1. Reproduce the paper number on the committed held-out test set

In [ ]:
!PYTHONPATH=. python train_verifier_reward.py \
  --eval-checkpoint {MODEL} --test-file benchmark_test.jsonl

## 2. Freshly generated unseen test set (new seed, deduplicated vs training)

In [ ]:
# Rebuild the exact corpus the released model was trained on, then generate a
# fresh corpus from a NEW seed and drop any trace whose (root, chain, action)
# decision context overlaps training or the committed test. What remains is
# genuinely unseen, same-distribution data.
from trace_benchmark import generate_corpus, write_jsonl, load_traces
from make_expanded_train import corpus_canonicals, drop_overlapping, label_stats

test_canon = corpus_canonicals(load_traces('benchmark_test.jsonl'))
a, b = generate_corpus(101, 150)                      # released model's train draw
train_canon = corpus_canonicals(drop_overlapping(a + b, test_canon, 'train'))

UNSEEN_SEED = 999
c, d = generate_corpus(UNSEEN_SEED, 60)
unseen = drop_overlapping(c + d, train_canon | test_canon, 'unseen')
uc = corpus_canonicals(unseen)
assert not (uc & train_canon) and not (uc & test_canon), 'leakage!'
write_jsonl(unseen, 'unseen_test.jsonl')
print('unseen_test.jsonl:', label_stats(unseen))

In [ ]:
!PYTHONPATH=. python train_verifier_reward.py \
  --eval-checkpoint {MODEL} --test-file unseen_test.jsonl